# 0x0A Data Science for Hunting
Detect DGA domains using Shannon Entropy and K-Means.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

def shannon_entropy(s):
    if not s:
        return 0
    p, lns = pd.Series(list(s)).value_counts() / len(s), float(len(s))
    return -sum(p * p.apply(math.log2))

# Sample Domain Data: Legitimate vs DGA (Domain Generation Algorithm)
domains = [
    "google.com", "microsoft.com", "apple.com", "amazon.com", "github.com",
    "asdfqwerzxcv.com", "xjqkwleb.net", "1a2b3c4d5e.org", "zxcvbnmasdfg.info", "qweasdzxc123.biz"
]
df = pd.DataFrame({"domain": domains})
df['entropy'] = df['domain'].apply(lambda x: shannon_entropy(x.split('.')[0]))
df['length'] = df['domain'].apply(lambda x: len(x.split('.')[0]))

display(df.head())


In [ ]:
# Cluster domains using K-Means to separate Legitimate from DGA
X = df[['entropy', 'length']]
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

# Anomaly detection using Isolation Forest
iso_forest = IsolationForest(contamination=0.4, random_state=42)
df['anomaly_score'] = iso_forest.fit_predict(X)

# Plotting the KMeans Clusters
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['length'], df['entropy'], c=df['cluster'], cmap='coolwarm', s=100, edgecolor='k')
plt.xlabel("Domain Length")
plt.ylabel("Shannon Entropy")
plt.title("DGA Detection via K-Means Clustering")

# Annotate points
for i, txt in enumerate(df['domain']):
    plt.annotate(txt, (df['length'][i] + 0.15, df['entropy'][i] - 0.05), fontsize=8)

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
